# Decoded Quantum Interferometry (DQI) with MindQuantum

本 notebook 展示如何使用 [MindQuantum](https://www.mindspore.cn/mindquantum) 框架实现
**Decoded Quantum Interferometry (DQI)** 算法，求解 max-XORSAT 组合优化问题。

**论文引用 (Paper reference):**
> Sabater, El Harzli, Besjes et al. "Towards solving industrial integer linear programs with Decoded Quantum Interferometry." arXiv:2509.08328 (2025).

## 算法概述

DQI 是一种量子优化算法，其核心思想是：

1. 将组合优化问题（max-XORSAT）编码为量子态的振幅
2. 通过多项式度 ℓ 控制采样偏置，使高目标函数值的状态被更大概率采样
3. 利用量子干涉和解码技术（LUT 或 BP1）实现高效求解

### max-XORSAT 问题

给定 $m \times n$ 二进制矩阵 $B$ 和右端向量 $\mathbf{v}$，找到 $n$ 位字符串 $\mathbf{x}$ 使得满足最多约束：

$$B\mathbf{x} = \mathbf{v} \pmod{2}$$

目标函数：$f(\mathbf{x}) = \sum_{i=1}^{m} (-1)^{(B\mathbf{x})_i + v_i}$，取值范围 $[-m, m]$。


## 1. 环境准备

安装依赖：

```bash
pip install mindquantum>=0.9.0 numpy scipy matplotlib tqdm

```


In [ ]:
# pylint: disable=missing-module-docstring, wrong-import-position, redefined-outer-name, invalid-name
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# 确保 dqi_mindquantum 包可导入
# 请从 dqi_mindquantum 目录运行此 notebook: jupyter notebook main.ipynb
sys.path.insert(0, os.getcwd())

from classical.bp_decoders import bp1_decode
from classical.dqi_expectation import compute_optimal_w, expected_constraints_dqi
from quantum.full_circuit import DQICircuit, run_dqi_histogram

print("All modules imported successfully.")


## 2. 问题定义

我们使用论文公式 (72) 中的固定 $8 \times 6$ max-XORSAT 实例：

- **8 个约束** (m=8)
- **6 个变量** (n=6)
- **多项式度 ℓ=2**

量子比特数 = $m + n + \lceil \log_2(\ell+1) \rceil = 8 + 6 + 2 = 16$（LUT 解码）


In [ ]:
# pylint: disable=redefined-outer-name
# 定义 max-XORSAT 实例: Bx = v (mod 2)
B = np.array(
    [
        [1, 1, 0, 0, 0, 1],
        [0, 1, 1, 0, 1, 0],
        [1, 0, 0, 1, 0, 0],
        [0, 0, 1, 1, 1, 0],
        [1, 1, 1, 0, 0, 0],
        [0, 1, 0, 1, 0, 1],
        [1, 0, 1, 0, 1, 0],
        [0, 0, 0, 1, 1, 1],
    ],
    dtype=int,
)

v = np.array([1, 0, 1, 1, 0, 1, 0, 1], dtype=int)

m, n = B.shape
ell = 2
print(f"Problem: {m}x{n} max-XORSAT, ell={ell}")
print(f"B =\n{B}")
print(f"v = {v}")


## 3. 随机采样基线

均匀随机采样作为基线：从 $\{0,1\}^n$ 中均匀采样，计算目标函数 $f(\mathbf{x})$。


In [ ]:
# pylint: disable=redefined-outer-name
n_samples = 100


def objective_function(x):
    """计算目标函数 f(x) = sum((-1)^(Bx+v))."""
    return int(np.sum((-1) ** ((B @ x + v) % 2)))


# 随机采样
np.random.seed(42)
random_samples = np.random.randint(0, 2, size=(n_samples, n))
f_x_random = [objective_function(s) for s in random_samples]

print(f"Random sampling ({n_samples} samples):")
print(f"  Average f(x) = {np.mean(f_x_random):.3f}")
print(f"  Max f(x)     = {np.max(f_x_random)}")
print(f"  f(x)>=3 ratio = {np.mean(np.array(f_x_random) >= 3):.3f}")


## 4. 经典 DQI 理论期望

在运行量子电路之前，我们先计算经典 DQI 理论期望值 $\langle S \rangle / m$。

核心步骤：

1. 计算最优权重 $w_k$（三对角矩阵的主特征向量）
2. 构建 A 矩阵
3. 计算 $\langle w | A | w \rangle$ 期望


In [ ]:
# pylint: disable=redefined-outer-name
# 计算最优权重
w_k = compute_optimal_w(m, ell)
print(f"Optimal weights w_k = {w_k}")

# 计算经典 DQI 期望值 (使用 BP1 解码器, T=5 次迭代)
config = {"max_iter": 5, "n_jobs": 1, "max_patterns": 500}
result = expected_constraints_dqi(
    B, v, ell, bp1_decode, config=config
)
print("\nClassical DQI expectation (BP1, T=5):")
print(f"  <S>/m = {result['s_normalized']:.4f}")


## 5. 量子电路构建与模拟

使用 `DQICircuit` 类构建完整 DQI 量子电路：

1. **权重编码** (WeightedUnaryEncoding)：在 error register 上制备 $w_k$ 叠加态
2. **Dicke 态制备** (UnkStatePreparation)：生成 $|D_k^m\rangle$ 态
3. **相位 Z 编码**：应用 $(-1)^{v_i}$ 相位
4. **伴随式计算**：CNOT 网络计算 $s = B^T y \mod 2$
5. **LUT 解码**：查表将 error register 反计算为 $|0\rangle$
6. **Hadamard 变换**：在 syndrome register 上应用 H 门

**后选择 (Post-selection)**：仅保留 error register = $|0\rangle$ 的结果。


In [ ]:
# 构建 DQI 电路
dqi = DQICircuit(B, v, ell=ell)
circ = dqi.build_circuit()

# 电路信息
summary = dqi.get_circuit_summary(circ)
print("Circuit summary:")
print(f"  Qubits: {summary['n_qubits']}")
print(f"  Gates:  {summary['n_gates']}")
print(f"  Depth:  {summary['depth']}")


In [ ]:
# 运行 DQI 量子电路模拟
result = run_dqi_histogram(B, v, ell=ell, shots=n_samples, seed=42)

print(f"DQI sampling ({n_samples} shots, post-selected):")
print(f"  Average satisfied constraints = {result['avg_s']:.3f}")
print(f"  <S>/m = {result['s_normalized']:.3f}")
print(f"  Post-selection fraction = {result['post_fraction']:.3f}")


## 6. 结果可视化

对比随机采样与 DQI 采样的目标函数分布。


In [ ]:
# pylint: disable=redefined-outer-name
# 可视化 DQI 约束满足分布
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(m + 1)
width = 0.35

# 随机采样分布
s_hist_random = np.zeros(m + 1)
for f_val in f_x_random:
    s_val = (m + f_val) // 2
    s_hist_random[s_val] += 1
s_hist_random /= len(f_x_random)

ax.bar(
    x - width / 2,
    s_hist_random,
    width,
    label="Random",
    color="steelblue",
    edgecolor="white",
)
ax.bar(
    x + width / 2,
    result["s_hist"],
    width,
    label="DQI",
    color="coral",
    edgecolor="white",
)

ax.set_xlabel("Number of Satisfied Constraints $s$", fontsize=13)
ax.set_ylabel("Probability", fontsize=13)
ax.set_title("DQI vs Random Sampling", fontsize=14)
ax.set_xticks(x)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# 汇总表格
print(f"\n{'Method':<20} {'<S>/m':<12}")
print("-" * 32)
print(f"{'Random':<20} {np.mean(f_x_random) / m + 0.5:<12.3f}")
print(f"{'DQI (ell=' + str(ell) + ')':<20} {result['s_normalized']:<12.3f}")


## 7. 资源扩展性分析

分析 DQI 量子比特数如何随问题规模 $(m, n)$ 扩展：

- **LUT 解码**: $N = m + n$
- **BP1 解码**: $N = 3m + n$（含辅助比特）

两者均为 $\mathcal{O}(m + n)$ 线性扩展，与 ℓ 无关。


In [ ]:
# pylint: disable=redefined-outer-name
sizes = [(m_val, int(m_val * 0.5)) for m_val in range(10, 210, 20)]
m_vals = [m_val * n_val for m_val, n_val in sizes]
qubits_lut = [m_val + n_val for m_val, n_val in sizes]
qubits_bp = [3 * m_val + n_val for m_val, n_val in sizes]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(m_vals, qubits_lut, "o-", color="#4c72b0", label="LUT: $N = m + n$")
ax.plot(m_vals, qubits_bp, "s-", color="#c44e52", label="BP1: $N = 3m + n$")
ax.set_xlabel("Problem Size $m \\times n$", fontsize=12)
ax.set_ylabel("Qubit Count", fontsize=12)
ax.set_title("Qubit Scaling: $\\mathcal{O}(m + n)$", fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 8. 运行完整实验

本项目还提供了复现论文图表的实验脚本：

```bash

# 复现 Fig 11: DQI 性能缩放曲线
python experiments/reproduce_fig11.py

# 复现 Fig 15: DQI 量子直方图
python experiments/reproduce_fig15.py

# 资源扩展性分析
python experiments/scaling_analysis.py

```

详见 `experiments/` 目录。


## 总结

- DQI 算法通过量子干涉和解码技术，显著优于均匀随机采样
- 量子比特数为 $\mathcal{O}(m+n)$，线性扩展
- LUT 解码适用于小规模问题（$n \leq 10$），BP1 解码适用于更大规模
- 多项式度 ℓ 控制采样偏置，通常 ℓ=1,2,3 即可获得良好性能

### 参考文献

1. Sabater, El Harzli, Besjes et al. "Towards solving industrial integer linear programs with Decoded Quantum Interferometry." arXiv:2509.08328 (2025).
2. MindQuantum: https://www.mindspore.cn/mindquantum
